In [0]:
%pip install pydantic-settings pyyaml pyrate_limiter -q

In [0]:
import pandas as pd 
import json
import logging
import sys 
from pathlib import Path
sys.path.insert(0, '/Workspace/Users/mhenning@modelpath.net/Dayforce_TAFW/src')
import requests
import pandas as pd
from pyspark.sql import functions as F


In [0]:
from tafw_ingest.config import Settings
from tafw_ingest.dayforce_client import DayforceApiError, DayforceClient
from tafw_ingest.department_map import DepartmentMap
from tafw_ingest.employees import fetch_employees_dataframe
from tafw_ingest.roster import (
    EMPLOYEE_EXPAND,
    department_xref_of,
    iter_in_scope_employees,
)

In [0]:

settings = Settings.from_env(config_path='/Workspace/Users/mhenning@modelpath.net/Dayforce_TAFW/conf/settings.dev.yaml')
client = DayforceClient.from_settings(settings)

# if client worked print successful or client auth failed
if client:
    print("Authentication Worked.")

else:
    print("Authenticating Failed.")

In [0]:
print("Step 1. Get all employee unique Ids.")
xrefs = client.list_employee_xrefs()
print("Done")

In [0]:
# COMMAND ----------
REPO_ROOT = "/Workspace/Users/mhenning@modelpath.net/Dayforce_TAFW"

# COMMAND ----------
# MAGIC %pip install -r {REPO_ROOT}/requirements.txt
# MAGIC %pip install -e {REPO_ROOT}

# COMMAND ----------
dbutils.library.restartPython()

# COMMAND ----------
import os
import subprocess
import sys

REPO_ROOT = "/Workspace/Users/mhenning@modelpath.net/Dayforce_TAFW"

os.environ["DAYFORCE_USERNAME"] = dbutils.secrets.get("dayforce", "username")
os.environ["DAYFORCE_PASSWORD"] = dbutils.secrets.get("dayforce", "password")

result = subprocess.run(
    [sys.executable, "tests/test.py"],
    cwd=REPO_ROOT,
    env=os.environ,
)
print(f"exit code: {result.returncode}")

In [0]:
# Credentials — store these in Databricks secrets, not hardcoded
company    = "bluedrop"  # your Dayforce namespace
report_name = "TAFW_Report"  # exact name as saved in Report Writer
username   = dbutils.secrets.get("dayforce", "username")
password   = dbutils.secrets.get("dayforce", "password")

base_url = f"https://cantest262.dayforcehcm.com/api/{company}/V1/Reports/{report_name}"

print(base_url)

In [0]:

# Connection config
base_url   = "https://cantest262.dayforcehcm.com/api/bluedrop/v1"
xref_code  = "svc_celigo_integration"
username   = dbutils.secrets.get(scope="dayforce", key="username")
password   = dbutils.secrets.get(scope="dayforce", key="password")

# Pull report data
response = requests.get(
    f"{base_url}/ReportMetadata/{xref_code}/Data",
    auth=(username, password),
    headers={"Accept": "application/json"}
)
response.raise_for_status()

payload = response.json()

# Inspect the raw structure first
print(payload.keys())
print(type(payload["Data"]))